## Model of "A Programmable Heterogeneous Microprocessor Based on Bit-Scalable In-Memory Computing", JSSC 2020

Paper by Hongyang Jia, Hossein Valavi, Yinqi Tang, Jintao Zhang, and Naveen Verma

### Description of The Macro

This macro partitions a large SRAM array into a 4x4 grid of subarrays that may be
indepently power gated. Additionally, it connects every group of three columns to reuse
outputs rather than inputs.

Inputs and outputs are both sliced into 1b slices, and the macro may sum the results of
multiple slices to produce a final output using variable-precision inputs and weights.

### Macro Level

- **Input Path**: Inputs are passed through zero gating before being XNOR-encoded and
  sent to row drivers. Row drivers support 1b input slices, so N-bit inputs are
  processed in N+1 cycles (XNOR adds an extra bit).
- **Weight Path**: Weight drivers are used to rewrite weights in the array.
- **Output Path**: Column drivers activate array columns to read analog outputs, which
  are then converted to digital using an 8b ADC. After the ADC, outputs are accumulated
  in a shift-add that sums output results across different input and weight slices.
  Finally, an output datapath performs quantization and activation functions on the
  outputs.

Next, there are four column groups in the macro, and each column group contains 64
column subgroups. Column groups may be independently power gated. Inputs are reused
between column groups and column subgroups.

### Column Subgroup Level 
 
There are 3 folded columns in a column subgroup. Folded columns, unlike standard
columns, reuse outputs rather than inputs. Beyond this, there are no additional
components in a column subgroup.

### Folded Column Level

- *Input Path*: Each input is passed directly to a row group in the folded column.
- *Weight Path*: A column bandwidth limiter sets the read and write bandwidth of each
  array column. Each weight is then passed to a row in the column.
- *Output Path*: A column bandwidth limiter sets the read and write bandwidth of each
  array column. One output is reused between rows in the column.

Inside each folded column, four row groups of 64 rows each reuse outputs. Row groups may
be independently power gated.

### Row Level

Each row contains a CiM unit that is composed of a SRAM cell and a capacitor to perform
analog MAC operations. The CiM unit uses a 1x1x8 (1b input slice x 1b weight slice x 8b
output) virtualized MAC unit to compute the MAC operation.

In [ ]:
from _scripts import *

display_important_variables("jia_jssc_2020")
get_spec("jia_jssc_2020").arch

### Area Breakdown

This test replicates the results of Fig. 11 in the paper. We show the area of the ADC,
CiM, NMC data path, and sparsity controller.

In [ ]:
spec = get_spec("jia_jssc_2020")
evaluated = spec.calculate_component_costs()

In [ ]:
groups = {
    "ADC": ["ADC"],
    "CiM": [
        "RowDrivers",
        "WeightDrivers",
        "CimUnit",
        "ColumnDrivers",
        "BitcellCapacitor",
    ],
    "NMC Data Path": ["OutDatapath", "ShiftAdd"],
    "Sparsity Controller": ["InputZeroGating"],
}
ref = {
    "ADC": 0.497e6,
    "CiM": 2.91e6,
    "NMC Data Path": 0.497e6,
    "Sparsity Controller": 0.392e6,
}
area = get_area_breakdown("jia_jssc_2020")
modeled = {k: sum(area[c] for c in cs) * 1e12 for k, cs in groups.items()}

fig, ax = plt.subplots(figsize=(8, 6))
bar_comparison(
    {"reference": ref, "model": modeled},
    "Component",
    "Area (um^2)",
    "Area Breakdown",
    ax,
)
plt.tight_layout()
plt.show()
for k in ref:
    print(f"{k:<22} {modeled[k]:10.0f}  {ref[k]:10.0f}  {modeled[k]/ref[k]:.2f}x")

### Energy Breakdown

This test replicates the results of Table I in the paper.

We show the area and energy of the macro at 0.85V and 1.2V power supplies using 1b
inputs and weights. We will report the energy of the ADC, CiM, and NMC data path.

We see that increasing the voltage from 0.85V to 1.2V increases the energy consumption
of each component of the macro.

In [ ]:
N = 256
groups = {
    "adc": ["ADC"],
    "CiM": [
        "RowDrivers",
        "WeightDrivers",
        "CimUnit",
        "BitcellCapacitor",
        "ColumnDrivers",
    ],
    "NMC Data Path": ["OutDatapath", "ShiftAdd"],
}
ref = {
    0.85: {"adc": 1.79 * N, "CiM": 9.7 * N, "NMC Data Path": 8.3 * N},
    1.2: {"adc": 3.56 * N, "CiM": 20.4 * N, "NMC Data Path": 14.7 * N},
}


def _run(v):
    spec = get_spec("jia_jssc_2020", add_dummy_main_memory=True)
    spec.variables.voltage = v
    spec.variables.input_bits = 1
    spec.variables.weight_bits = 1
    spec.variables.output_bits = 1
    for e in spec.workload.einsums:
        for ta in e.tensor_accesses:
            ta.bits_per_value = 1
    return v, spec.map_workload_to_arch(print_progress=False)


results = dict(parallel([delayed(_run)(v) for v in [0.85, 1.2]]))


def _per_array_pj(r):
    pc_e = r.energy(per_component=True)
    return {
        k: sum(float(pc_e[c]) for c in cs if c in pc_e) * 1e12
        for k, cs in groups.items()
    }


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (v, r) in zip(axes, results.items()):
    bar_comparison(
        {"reference": ref[v], "model": _per_array_pj(r)},
        "Component",
        "Energy (pJ/Full Array 1b MACs)",
        f"Energy Breakdown at {v}V",
        ax,
    )
plt.tight_layout()
plt.show()

for v, r in results.items():
    pc = r.per_compute()
    print(
        f"{v}V  TOPS={2/pc.latency()/1e12:.4f}  TOPS/W={2/pc.energy()/1e12:.1f}  fJ/MAC={pc.energy()*1e15:.1f}"
    )

### Energy Efficiency, Throughput, and Compute Density

This test replicates the results of Table II in the paper.

We show the area, energy efficiency, and throughput of the macro at 0.85V and 1.2V power
supplies using 1b inputs and weights.

We see that increasing the voltage from 0.85V to 1.2V increases throughput and compute
density at the cost of increased the energy consumption.

In [ ]:
tops_components = [
    "RowDrivers",
    "CimUnit",
    "ADC",
    "ColumnDrivers",
    "BitcellCapacitor",
    "WeightDrivers",
]
ref = {0.85: (0.874, 400, 0.24), 1.2: (2.185, 192, 0.6)}


def _run(v):
    spec = get_spec("jia_jssc_2020", add_dummy_main_memory=True)
    spec.variables.voltage = v
    spec.variables.input_bits = spec.variables.weight_bits = (
        spec.variables.output_bits
    ) = 1
    for e in spec.workload.einsums:
        for ta in e.tensor_accesses:
            ta.bits_per_value = 1
    r = spec.map_workload_to_arch(print_progress=False)
    ev = spec.calculate_component_costs()
    e_tops = sum(
        float(r.per_compute().energy(per_component=True)[c]) for c in tops_components
    )
    a_tops = sum(float(ev.arch.per_component_total_area[c]) for c in tops_components)
    pc = r.per_compute()
    for k2, v2 in pc.latency(per_component=True).items():
        print(f"{k2}: {v2:.2e} s")
    return (
        v,
        2 / pc.latency() / 1e12,
        2 / e_tops / 1e12,
        (2 / pc.latency() / 1e12) / (a_tops * 1e6),
    )


out = parallel([delayed(_run)(v) for v in [0.85, 1.2]])
labels = [str(v) for v, *_ in out]
metrics = [
    ("Throughput (1b TOPS)", "Voltage vs. Throughput", 0, 1),
    ("Energy Efficiency (1b TOPS/W)", "Voltage vs. Energy Efficiency", 1, 2),
    ("Compute Density (1b TOPS/mm^2)", "Voltage vs. Compute Density", 2, 3),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (yl, title, ref_i, mod_i) in zip(axes, metrics):
    bar_comparison(
        {
            "reference": dict(zip(labels, [ref[v][ref_i] for v, *_ in out])),
            "model": dict(zip(labels, [m[mod_i] for m in out])),
        },
        "Voltage (V)",
        yl,
        title,
        ax,
    )
plt.tight_layout()
plt.show()

### Throughput versus Number of Input and Weight Bits

This test replicates the results of Fig. 10 C_cimu in the paper.

We show the area and throughput of the macro with 1, 2, 4, and 8b inputs and weights.
For each configuration, we measure the throughput of the macro in TOPS.

We see that the this macro can flexibly trade off bit precision and throughput. It does
so by computing with 1b slices of inputs and weights, then shifting + adding together
results from a variable number of slices. Each input slice is processed in a different
timestep, so fewer (more) input slices will decrease (increase) latency. Each weight
slice is stored in a different column, so fewer (more) weight slices will decrease
(increase) storage density.

There is a slight discrepancy at the highest-bitwidth result, where the throughput of
the published macro decreases more than expected. This is due to bottlenecks in the
published macro's memory hierarchy.

In [ ]:
def _run(input_bits):
    spec = get_spec("jia_jssc_2020", add_dummy_main_memory=True)
    spec.variables.input_bits = input_bits
    spec.variables.weight_bits = spec.variables.output_bits = 1
    for e in spec.workload.einsums:
        for ta in e.tensor_accesses:
            ta.bits_per_value = input_bits if ta.name == "input" else 1
    r = spec.map_workload_to_arch(print_progress=False)
    return input_bits, 2 / r.per_compute().latency() / 1e12


out = parallel([delayed(_run)(b) for b in [1, 2, 4, 8]])
labels = [str(b) for b, _ in out]
ref = [0.874 / d for d in [1, 2, 4, 600 / 64]]

fig, ax = plt.subplots(figsize=(8, 5))
bar_comparison(
    {
        "reference": dict(zip(labels, ref)),
        "model": dict(zip(labels, [t for _, t in out])),
    },
    "Number of Input and Weight Bits",
    "Throughput (TOPS)",
    "Input and Weight Bits vs. Throughput",
    ax,
)
plt.tight_layout()
plt.show()

### Exploration of Inter-Column Output Reuse Versus Energy Efficiency

This test explores how the macro's column folding strategy impacts the energy efficiency
and throughput of a DNN. Column folding connects columns together to share outputs,
rather than inputs as is done in non-folded columns.

We can see that, as the number of folded columns increases, the energy due to output
processing (column readout, ADC) decreases because folding columns increases output
reuse and allows output readout circuitry energy to be amortized across more
computations (*e.g.,* one ADC can read the results from more than column at a time
instead of just one). However, the energy due to input processing (DAC, row drivers)
increases because amount of input reuse decreases leading to more DAC converts and input
driver activations.

In [ ]:
from tqdm import tqdm
import accelforge as af

n_macros = 4096
groups = {
    "Output Processing": ["ADC", "ColumnDrivers", "OutDatapath", "ShiftAdd"],
    "Input Processing": ["RowDrivers", "InputZeroGating"],
    "Other": ["CimUnit", "BitcellCapacitor"],
}


def _set_folded(spec, n_folded):
    for n in spec.arch.nodes:
        if not hasattr(n, "spatial"):
            continue
        for s in n.spatial:
            if s.name == "subgroup_ARRAY_COLUMNS":
                s.fanout = 192 // n_folded
            elif s.name == "folded_columns_ARRAY_COLUMNS":
                s.fanout = n_folded


def _max_util(n_folded):
    spec = get_spec("jia_jssc_2020", add_dummy_main_memory=True)
    _set_folded(spec, n_folded)
    spec.workload.rank_sizes = {
        "M": 1,
        "N": 4 * (192 // n_folded),
        "K": n_folded * 4 * 192,
    }
    return spec.map_workload_to_arch(print_progress=False)


def _resnet18(n_folded):
    spec = get_spec("jia_jssc_2020", add_dummy_main_memory=True)
    _set_folded(spec, n_folded)
    spec.workload = af.Workload.from_yaml(
        af.examples.workloads.compute_in_memory.resnet18,
        top_key="workload",
        jinja_parse_data={"BATCH_SIZE": 1},
    )
    spec.mapper.metrics = af.Metrics.ENERGY
    spec.mapper.explore_loop_orders = False
    return spec.map_workload_to_arch(print_progress=False)


def _grouped(r, scale):
    e = {k: float(v) for k, v in r.energy(per_component=True).items()}
    return {k: sum(e[c] for c in cs) * scale for k, cs in groups.items()}


max_util_results = {}
resnet_results = {}
for n_folded in tqdm(range(1, 9)):
    max_util_results[n_folded] = _max_util(n_folded)
    resnet_results[n_folded] = _resnet18(n_folded)

mu_uj = {x: _grouped(r, 1e6 * n_macros) for x, r in max_util_results.items()}
rn_mj = {x: _grouped(r, 1e3) for x, r in resnet_results.items()}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
bar_stacked(
    mu_uj,
    "Number of Columns Reusing Outputs",
    f"Energy (uJ per {n_macros}-Macro System Pass)",
    "Max-Utilization Energy vs. # Columns Reusing Outputs",
    axes[0],
)
bar_stacked(
    rn_mj,
    "Number of Columns Reusing Outputs",
    "Energy (mJ per DNN Run)",
    "ResNet18 Energy vs. # Columns Reusing Outputs",
    axes[1],
)
plt.tight_layout()
plt.show()